In [33]:
import pandas as pd
import glob
csv_files = glob.glob(
    r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\*.csv"
)

datasets = {}

for file in csv_files:
    datasets[file] = pd.read_csv(file)

In [34]:
orders = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\olist_orders_dataset.csv"]
geolocation = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\olist_geolocation_dataset.csv"]
products = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\olist_products_dataset.csv"]
order_reviews = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\olist_order_reviews_dataset.csv"]
order_payments = datasets[r"C:\Users\Abdul\Documents\VS_code_projects\OLIST_Data_Cleaning\data\raw\olist_order_payments_dataset.csv"]


### MY ASSESSMENTS 


| Issue                         | Finding                                                        | Decision            | Reason                                                               |
| ----------------------------- | -------------------------------------------------------------- | ------------------- | -------------------------------------------------------------------- |
| Geolocation exact duplicates  | 261,831 exact duplicate rows                                   | Remove              | Duplicates contain no additional information                         |
| Date/timestamp columns        | Stored as strings                                              | Convert to datetime | Correct data type for temporal analysis                              |
| Product column names          | `product_name_lenght`, `product_description_lenght` misspelled | Rename              | Standardize column names                                             |
| Delivered orders              | 8 delivered orders missing customer delivery date              | Preserve            | Actual dates cannot be reliably determined                           |
| Repeated review IDs           | 789 review IDs occur multiple times                            | Preserve            | Different `order_id` values could represent meaningful relationships |
| Zero credit-card installments | 2 records have 0 installments with nonzero payment values      | Preserve            | Meaning of `0` cannot be established reliably                        |
| Missing product categories    | 610 products have missing categories                           | Preserve            | Actual categories cannot be determined                               |
| Missing product dimensions    | 2 products missing all physical measurements                   | Preserve            | Cannot reliably reconstruct measurements                             |
| Geographic inconsistencies    | Potential ZIP/state and coordinate anomalies                   | Preserve            | No authoritative source available for correction                     |
| Untranslated categories       | 2 categories lack English translations                         | Preserve            | Do not invent translations                                           |


### Removing Geolocation Duplicates


In [35]:

geolocation = geolocation.drop_duplicates()
geolocation.duplicated().sum()


np.int64(0)

### Converting Order Date columns from Strings to DateTime

In [36]:
date_columns = ["order_purchase_timestamp", "order_approved_at", 'order_delivered_carrier_date', "order_delivered_customer_date", "order_estimated_delivery_date"]

orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [37]:
orders[date_columns] = orders[date_columns].apply(pd.to_datetime)

orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

### Misspelled Product Column Names

In [38]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [39]:
products = products.rename(columns = {
    "product_name_lenght" : "product_name_length",
    "product_description_lenght" : "product_description_length" 
})

products.columns

Index(['product_id', 'product_category_name', 'product_name_length',
       'product_description_length', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

### Date Columns in Reviews Dtype Conversion

In [40]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                99224 non-null  str  
 1   order_id                 99224 non-null  str  
 2   review_score             99224 non-null  int64
 3   review_comment_title     11568 non-null  str  
 4   review_comment_message   40977 non-null  str  
 5   review_creation_date     99224 non-null  str  
 6   review_answer_timestamp  99224 non-null  str  
dtypes: int64(1), str(6)
memory usage: 5.3 MB


In [44]:
review_date_columns = ["review_creation_date", "review_answer_timestamp"]

order_reviews[review_date_columns] = order_reviews[review_date_columns].apply(pd.to_datetime)

order_reviews.dtypes

review_id                             str
order_id                              str
review_score                        int64
review_comment_title                  str
review_comment_message                str
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

### Cleaning Validation Check

In [45]:
print("Geolocation duplicates:", geolocation.duplicated().sum())

print("\nOrder date types:")
print(orders[date_columns].dtypes)

print("\nReview date types:")
print(order_reviews[review_date_columns].dtypes)

print("\nProduct columns:")
print(products.columns.tolist())

Geolocation duplicates: 0

Order date types:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

Review date types:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object

Product columns:
['product_id', 'product_category_name', 'product_name_length', 'product_description_length', 'product_photos_qty', 'product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']
